# Inching v1.1 on Google Colab

GPU normal mode analysis of large macromolecular assemblies.
Lam, Nakano & Katritch, *Nat Commun* **15**, 3479 (2024). https://github.com/jhmlam/Inching

Runtime → Change runtime type → **GPU** (T4 is enough for the example). Run the cells top to bottom.

In [ ]:
#@title 1. Install (about 1 minute)
!pip install -q openmm mdtraj
!pip install -q --no-deps git+https://github.com/jhmlam/Inching.git@v1.1.1
import importlib.metadata, torch, cupy
print('inching', importlib.metadata.version('inching'), '| torch', torch.__version__, '| cupy', cupy.__version__, '| GPU', torch.cuda.get_device_name(0))

In [ ]:
#@title 2. Structure: upload a pdb/cif, or leave empty to use ubiquitin 1UBQ
import os
from google.colab import files
uploaded = files.upload()
if uploaded:
    structure = list(uploaded)[0]
else:
    structure = '1ubq.pdb'
    os.system('wget -q -O 1ubq.pdb https://files.rcsb.org/download/1UBQ.pdb')
print('using', structure)

In [ ]:
#@title 3. Parameters
n_modes = 16  #@param {type:"integer"}
solver = 'trlmhd'  #@param ['trlmhd', 'jdmhd', 'irlmhd']
rc_angstrom = 8.0  #@param {type:"number"}
int64_indexing = False  #@param {type:"boolean"}

from inching.cli import load, info
xyz = load(structure)
print({k: v for k, v in info(xyz, rc_angstrom, 100, n_modes).items() if k in ('n_atoms', 'nnz_lower_est', 'int32_ok', 'gb_hessian_int32', 'gb_hessian_int64')})

In [ ]:
#@title 4. Compute modes
import time, json
import numpy as np
from inching.cli import run_nma
t0 = time.time()
eigval, eigvec, residual, gpu_bytes = run_nma(xyz, k=n_modes, solver=solver, rc=rc_angstrom, int64=int64_indexing)
stem = os.path.basename(structure).rsplit('.', 1)[0]
os.makedirs('Result', exist_ok=True)
np.save(f'Result/{stem}_eigval.npy', eigval)
np.save(f'Result/{stem}_eigvec.npy', eigvec)
print('%d modes in %.1f s, max residual %.1e, GPU pool %.2f GB' % (len(eigval), time.time() - t0, max(residual), gpu_bytes / 1024**3))
print('eigenvalues:', np.array2string(eigval, precision=4))

In [ ]:
#@title 5. Animate modes and download (multi-model cif, B-factor = displacement magnitude)
modes = '0-5'  #@param {type:"string"}
amplitude_angstrom = 2.0  #@param {type:"number"}

import shutil
from inching.cli import animate, parse_modes
for fn in animate(structure, eigvec, parse_modes(modes, eigvec.shape[0]), 'Result', 7, amplitude_angstrom, 'cif'):
    print(fn)
shutil.make_archive(f'InchingResult_{stem}', 'zip', 'Result')
files.download(f'InchingResult_{stem}.zip')

In [ ]:
#@title 6. The same from the command line (what you get with `conda install -c conda-forge inching`)
!inching info $structure -k $n_modes
!inching nma  $structure -k $n_modes --solver $solver --rc $rc_angstrom -o Result_cli
!inching animate $structure Result_cli/{stem}_eigvec.npy --modes 0-1 --amplitude $amplitude_angstrom -o Result_cli
!ls Result_cli

Outputs of `inching nma`: `<id>_eigval.npy` (k,), `<id>_eigvec.npy` (k, n_atoms, 3) in file atom order, `<id>_nma.json` (parameters, eigenvalues, residuals, runtime). `inching animate` writes one multi-model cif/pdb per mode; open them in PyMOL/ChimeraX as a trajectory, `spectrum b` colours by displacement. Python API: `inching.cli.run_nma`, `animate`, `info`.